In [21]:
import pandas as pd
import time

# requirements: pip install pulp

In [22]:
start=time.perf_counter()

In [23]:
#add this info to a settings dict?
current_gw = 9 #predictions from gw+1
current_season = 20262027
divx_team_code = 5049406 # My 26/27 divx team code

In [24]:
from DatabaseCreator import DatabaseCreator
import CJDH_local_settings

#Run Database Creator
if __name__ == "__main__":
    db_creator = DatabaseCreator(db_settings=CJDH_local_settings.local_settings['FPL_Points_Predictor'])
    fpl_engine = db_creator.get_engine_for("fpl_data_analysis")

In [25]:
from FPL_League_Data_Scraper import FPL_League_Data_Scraper

league_code = "1081956"

#Run Data Scraper for DivX league
if __name__ == "__main__":
    divx = FPL_League_Data_Scraper(league_code, current_season)
    divx_picksdf, membergw_summary = divx.scrape_picks()

#Add a new staging table & data into the database
divx_picks = divx_picksdf[['member', 'memberid','season', 'gw', 'pick', 
                 'element', 'position', 'multiplier', 
                 'is_captain', 'is_vice_captain']]

table_name = f"divx_picks_{current_season}"
if __name__ == "__main__":
    db_creator.create_staging_table_then_insert_data(table_name, data=divx_picks)
    divx_picks = db_creator.table_to_df(table_name=table_name)
    print(divx_picks.info())

Data fetched, league name:  DIV X
Fetched history for Stephen Collier (ID: 391266)
Fetched history for Joe Meader (ID: 2881244)
Fetched history for Ollie Carroll (ID: 4913097)
Fetched history for Christopher Harris (ID: 5049406)
Fetched history for Bernard Chalk (ID: 4184853)
Fetched history for Marcin Wichlinski (ID: 4010004)
Fetched history for Christopher Collier (ID: 4705527)
Fetched history for Callum Harling (ID: 3081288)
Fetched history for Jack Carroll (ID: 4355821)
Fetched history for Anthony Collier (ID: 5037473)
Fetched history for Christopher Harris (ID: 4486919)
Fetched history for Luiton Fernandes (ID: 3060165)
Fetched history for Samia Shelim (ID: 678874)
Fetched history for Ryan Boyce (ID: 4036890)
Fetched history for Tommy Davies (ID: 3340544)
Fetched history for Conner Harling (ID: 4484147)
Fetched history for Tajwar Shelim (ID: 4512122)
Fetched history for Christakis Christodoulou (ID: 5691733)
Fetched history for Omari Douglas-Rankine (ID: 8921685)
Max gameweek data

In [26]:
view = db_creator.run_sql(f"""
    WITH member_points AS (
        SELECT 
            member,
            memberid,
            SUM(
                COALESCE(playergw.total_points, 0) * COALESCE(divx_picks_{current_season}.multiplier, 0)) AS total_points   
        FROM divx_picks_{current_season}
        LEFT JOIN playergw 
            ON divx_picks_{current_season}.element = playergw.element
            AND divx_picks_{current_season}.season = playergw.season
            AND divx_picks_{current_season}.gw = playergw.event
        GROUP BY member, memberid
    )
    SELECT 
        member,
        memberid,
        total_points 
    FROM member_points
    ORDER BY total_points DESC
""")
view

,member,memberid,total_points
0,Stephen Collier,391266.0,615.0
1,Joe Meader,2881244.0,597.0
2,Ollie Carroll,4913097.0,569.0
3,Christopher Harris,5049406.0,568.0
4,Bernard Chalk,4184853.0,566.0
5,Marcin Wichlinski,4010004.0,563.0
6,Christopher Collier,4705527.0,557.0
7,Callum Harling,3081288.0,549.0
8,Jack Carroll,4355821.0,541.0
9,Anthony Collier,5037473.0,540.0


In [27]:
print(f"Best players for next 5 gameweeks with exponential decay (normalized): GW{current_gw+1} to GW{current_gw+5}")

# For weights to sum to 1 with decay_factor d and initial weight w:
# w * (1 + d + d^2 + d^3 + d^4) = 1
# So w = 1 / (1 + d + d^2 + d^3 + d^4)

decay_factor = 0.75 #  lower decay factor means more emphasis on nearer gws, 1 is equal weights.
weight_sum = sum([decay_factor**i for i in range(5)])
initial_weight = 1 / weight_sum # weights add to 1

view = db_creator.run_sql(f"""
    WITH predictions_status AS (
    SELECT *
    FROM predictions
    LEFT JOIN player_status ON predictions.element = player_status.id
    )
    SELECT
        player_name_id,
        status,
        in_my_team,
        position,
        value,
        team_name,
        ROUND(AVG(opp_team_elo),1) AS avg_opp_team_elo,
        ROUND(SUM(
            CASE 
                WHEN event = {current_gw+1} THEN xpoints * {initial_weight}
                WHEN event = {current_gw+2} THEN xpoints * {initial_weight * decay_factor}
                WHEN event = {current_gw+3} THEN xpoints * {initial_weight * decay_factor**2}
                WHEN event = {current_gw+4} THEN xpoints * {initial_weight * decay_factor**3}
                WHEN event = {current_gw+5} THEN xpoints * {initial_weight * decay_factor**4}
                ELSE 0
            END
        ), 2) AS total_xpoints_decayed,
        ROUND(AVG(mins_pred),1) AS avg_mins_pred,
        ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
        ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
        ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
    FROM predictions_status
    WHERE event BETWEEN {current_gw+1} AND {current_gw+5}
        AND status IN ('a', 'd')
        --AND position IN ('DEF')
        --AND VALUE <= 44
    GROUP BY player_name_id, status, position, value, team_name, in_my_team
    ORDER BY total_xpoints_decayed DESC
""")
view.head(50)

Best players for next 5 gameweeks with exponential decay (normalized): GW10 to GW14


,player_name_id,status,in_my_team,position,value,team_name,avg_opp_team_elo,total_xpoints_decayed,avg_mins_pred,avg_xgp90_pred,avg_xap90_pred,avg_xgcp90_pred
0,Florian Wirtz,a,0,MID,72.0,Liverpool,1049.2,5.91,67.8,0.719,0.175,1.189
1,Erling Haaland,a,1,FWD,158.0,Man City,1086.5,5.69,84.9,0.636,0.215,1.202
2,Bruno Borges Fernandes,a,0,MID,117.0,Man Utd,1083.5,5.28,80.6,0.361,0.272,1.444
3,Morgan Gibbs-White,a,1,MID,79.0,Nott'm Forest,1114.8,5.27,82.8,0.288,0.336,1.486
4,Nicolas Jackson,a,0,FWD,69.0,Aston Villa,1032.4,5.24,82.3,0.696,0.071,1.225
5,Francisco Evanilson de Lima Barbosa,a,0,FWD,63.0,Bournemouth,1037.5,5.23,79.0,0.518,0.304,1.031
6,Pascal Groß,a,0,MID,62.0,Brighton,1054.4,5.07,79.8,0.292,0.206,1.264
7,Morgan Rogers,a,0,MID,81.0,Chelsea,1043.5,5.00,85.2,0.311,0.155,1.230
8,Phil Foden,a,0,MID,68.0,Man City,1086.5,4.99,61.2,0.592,0.188,0.992
9,Marcus Tavernier,a,1,MID,65.0,Bournemouth,1037.5,4.97,81.0,0.270,0.260,1.192


In [28]:
mydivx_team_20262027 = 5049406
mydiv1_team_20262027 = 4486919

alpha = 0.5
# blending factor α (alpha) between 0 and 1, where:
# α = 0 → pure ownership-weighted (current behaviour)
# α = 1 → pure absolute points
# α = 0.5 → equal blend

ownership_weighted_predictions = db_creator.run_sql(f"""
    WITH member_points AS (
        SELECT 
            member,
            memberid,
            SUM(COALESCE(playergw.total_points, 0) * COALESCE(divx_picks_{current_season}.multiplier, 0)) AS total_points   
        FROM divx_picks_{current_season}
        LEFT JOIN playergw 
            ON divx_picks_{current_season}.element = playergw.element
            AND divx_picks_{current_season}.season = playergw.season
            AND divx_picks_{current_season}.gw = playergw.event
        GROUP BY member, memberid
    ),
    top_5_members AS (
        SELECT memberid
        FROM member_points
        WHERE memberid NOT IN ({mydivx_team_20262027}, {mydiv1_team_20262027})
        ORDER BY total_points DESC
        LIMIT 5
    ),
    my_team AS (
        SELECT DISTINCT element
        FROM divx_picks_{current_season}
        WHERE memberid = {divx_team_code}
            AND gw = {current_gw}
    ),
    latest_predictions AS (
        SELECT *
        FROM predictions
        LEFT JOIN player_status ON predictions.element = player_status.id
        WHERE prediction_timestamp = (
            SELECT MAX(prediction_timestamp)
            FROM predictions
        )
    ),
    current_divx_picks AS (
        SELECT element, SUM(multiplier) AS total_multiplier
        FROM divx_picks_{current_season}
        WHERE divx_picks_{current_season}.gw = {current_gw}
            AND memberid IN (SELECT memberid FROM top_5_members)
        GROUP BY element
    ),
    player_ownership AS (
        SELECT
            CASE WHEN mt.element IS NOT NULL THEN 1 ELSE 0 END AS in_my_team,
            p.element,
            p.player_name_id,
            p.status,
            p.now_cost / 10.0 AS value,
            p.position,
            p.team_name,
            p.event,
            p.xpoints,
            COALESCE(cdx.total_multiplier, 0)::DECIMAL / NULLIF((SELECT COUNT(*) FROM top_5_members), 0) AS ownership_ratio
        FROM latest_predictions p
        LEFT JOIN current_divx_picks cdx
            ON cdx.element = p.element
        LEFT JOIN my_team mt
            ON mt.element = p.element
        WHERE p.status IN ('a', 'd')
    ),
    aggregated_stats AS (
        SELECT 
            in_my_team,
            element, 
            player_name_id,
            status,
            value, 
            position, 
            team_name,
            SUM(xpoints) AS sum_xpoints,
            MAX(ownership_ratio) * 100.0 AS ownership_percentage,
            MAX(ownership_ratio) AS ownership_ratio,
            -- Points gained relative to top 5 if owned (0 when 100% owned, full xP when 0% owned)
            SUM(xpoints) * (1.0 - MAX(ownership_ratio)) AS differential_upside,
            -- Relative delta against rivals (-xP * O if not owned, xP * (1 - O) if owned)
            SUM(xpoints) * (in_my_team - MAX(ownership_ratio)) AS net_rank_swing
        FROM player_ownership
        WHERE event BETWEEN {current_gw + 1} AND 38
        GROUP BY in_my_team, element, player_name_id, status, value, position, team_name
    )
    SELECT 
        *,
        -- Scaled blend between raw total points and differential gain potential
        sum_xpoints * ({alpha} + (1.0 - {alpha}) * (1.0 - ownership_ratio)) AS blended_score
    FROM aggregated_stats
""")

print(f"BEST RANK MOVERS! Top 15 differential picks in my team for GW{current_gw + 1} - GW38:")
in_my_team_filter = ownership_weighted_predictions['in_my_team'] == 1
ownership_weighted_predictions[in_my_team_filter].sort_values(by='net_rank_swing', ascending=False)

BEST RANK MOVERS! Top 15 differential picks in my team for GW10 - GW38:


,in_my_team,element,player_name_id,status,value,position,team_name,sum_xpoints,ownership_percentage,ownership_ratio,differential_upside,net_rank_swing,blended_score
484,1,79.0,Francisco Evanilson de Lima Barbosa,a,6.4,FWD,Bournemouth,150.527095,0.0,0.0,150.527095,150.527095,150.527095
492,1,427.0,Bryan Mbeumo,a,7.9,MID,Man Utd,141.832394,0.0,0.0,141.832394,141.832394,141.832394
483,1,60.0,James Hill,a,5.5,DEF,Bournemouth,136.579646,0.0,0.0,136.579646,136.579646,136.579646
486,1,173.0,Bobby Thomas,d,4.0,DEF,Coventry City,112.906285,0.0,0.0,112.906285,112.906285,112.906285
487,1,175.0,Milan van Ewijk,a,3.9,DEF,Coventry City,103.622554,0.0,0.0,103.622554,103.622554,103.622554
494,1,497.0,Martin Dubravka,a,4.0,GK,Spurs,102.581997,0.0,0.0,102.581997,102.581997,102.581997
490,1,399.0,Rayan Cherki,a,7.9,MID,Man City,99.864794,0.0,0.0,99.864794,99.864794,99.864794
493,1,480.0,Morgan Gibbs-White,a,7.9,MID,Nott'm Forest,156.759116,40.0,0.4,94.055469,94.055469,125.407293
481,1,8.0,Riccardo Calafiori,d,5.9,DEF,Arsenal,127.909268,40.0,0.4,76.745561,76.745561,102.327414
480,1,1.0,David Raya Martín,a,6.2,GK,Arsenal,116.006210,40.0,0.4,69.603726,69.603726,92.804968


In [29]:
print(f"DONT MISS OUT! Most owned points scorers for GW{current_gw + 1} - GW38:")
ownership_weighted_predictions.sort_values(by='net_rank_swing', ascending=True).head(20)

DONT MISS OUT! Most owned points scorers for GW10 - GW38:


,in_my_team,element,player_name_id,status,value,position,team_name,sum_xpoints,ownership_percentage,ownership_ratio,differential_upside,net_rank_swing,blended_score
491,1,411.0,Erling Haaland,a,15.8,FWD,Man City,165.235340,200.0,2.0,-165.235340,-165.235340,0.000000
112,0,166.0,Nicolas Jackson,a,7.0,FWD,Aston Villa,151.235172,60.0,0.6,60.494069,-90.741103,105.864620
88,0,124.0,Pascal Groß,a,6.2,MID,Brighton,147.458524,60.0,0.6,58.983410,-88.475115,103.220967
23,0,40.0,Morgan Rogers,a,8.1,MID,Chelsea,144.094659,60.0,0.6,57.637864,-86.456796,100.866262
43,0,68.0,Marcus Tavernier,a,6.6,MID,Bournemouth,143.160017,40.0,0.4,85.896010,-57.264007,114.528014
78,0,112.0,Jan Paul van Hecke,a,5.0,DEF,Spurs,114.612836,40.0,0.4,68.767702,-45.845135,91.690269
305,0,449.0,Lewis Hall,a,5.3,DEF,Newcastle,107.609246,40.0,0.4,64.565547,-43.043698,86.087396
288,0,426.0,Bruno Borges Fernandes,a,11.7,MID,Man Utd,154.076181,20.0,0.2,123.260945,-30.815236,138.668563
10,0,15.0,Martin Ødegaard,a,6.7,MID,Arsenal,143.858819,20.0,0.2,115.087056,-28.771764,129.472937
367,0,542.0,Enzo Le Fée,a,5.7,MID,Sunderland,142.061200,20.0,0.2,113.648960,-28.412240,127.855080


In [30]:
print(f"Best differential picks not in my team for GW{current_gw + 1} - GW38:")
not_in_my_team_filter = ownership_weighted_predictions['in_my_team'] == 0
ownership_weighted_predictions[not_in_my_team_filter].sort_values(by='differential_upside', ascending=False).head(20)

Best differential picks not in my team for GW10 - GW38:


,in_my_team,element,player_name_id,status,value,position,team_name,sum_xpoints,ownership_percentage,ownership_ratio,differential_upside,net_rank_swing,blended_score
248,0,366.0,Florian Wirtz,a,7.2,MID,Liverpool,169.224437,0.0,0.0,169.224437,0.0,169.224437
274,0,398.0,Phil Foden,a,6.8,MID,Man City,145.639064,0.0,0.0,145.639064,0.0,145.639064
8,0,13.0,Declan Rice,d,7.4,MID,Arsenal,145.589812,0.0,0.0,145.589812,0.0,145.589812
7,0,11.0,Cristhian Mosquera,a,5.2,DEF,Arsenal,142.656494,0.0,0.0,142.656494,0.0,142.656494
42,0,67.0,Rayan Vitor Simplício Rocha,a,6.3,MID,Bournemouth,137.865384,0.0,0.0,137.865384,0.0,137.865384
386,0,566.0,António João Pereira de Albuquerque Tavares da...,a,5.0,DEF,Bournemouth,136.958953,0.0,0.0,136.958953,0.0,136.958953
177,0,261.0,Alex Iwobi,a,5.4,MID,Fulham,135.319529,0.0,0.0,135.319529,0.0,135.319529
73,0,106.0,Igor Thiago Nascimento Rodrigues,a,7.8,FWD,Brentford,134.545718,0.0,0.0,134.545718,0.0,134.545718
267,0,389.0,Matheus Nunes,a,5.9,DEF,Man City,134.434699,0.0,0.0,134.434699,0.0,134.434699
249,0,367.0,Cody Gakpo,a,7.1,MID,Liverpool,134.342417,0.0,0.0,134.342417,0.0,134.342417


In [31]:
# print(f"BLENDED SCORE: POINTS & RANK MOVERS! Top 15 differential picks in my team for GW{current_gw + 1} - GW38:")
# ownership_weighted_predictions.sort_values(by='blended_score', ascending=False).head(20)

In [32]:
# print(f"\nTop 20 predicted xPoints between {current_gw+1} and {current_gw+5}:")
# next5gw_predictions = db_creator.run_sql(f"""
#     WITH predictions_status AS (
#     SELECT *
#     FROM predictions
#     LEFT JOIN player_status ON predictions.element = player_status.id
#     )
#     SELECT
#         player_name_id,
#         status,
#         position,
#         team_name,
#         value/10 as value,
#         ROUND(AVG(opp_team_elo),1) AS avg_opp_team_elo,
#         ROUND(SUM(xpoints),2) AS total_xpoints,
#         ROUND(AVG(mins_pred),1) AS avg_mins_pred,
#         ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
#         ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
#         ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
#     FROM predictions_status
#     WHERE event BETWEEN {current_gw+1} AND {current_gw+5}
#         AND status IN ('a', 'd')
#         -- AND position = 'FWD'
#         -- AND value <= 45
#         -- AND team_name = 'Crystal Palace'
#     GROUP BY player_name_id, status, position, team_name, value
#     ORDER BY total_xpoints DESC
# """).dropna(subset=['value', 'total_xpoints'])
# next5gw_predictions.head(20)


In [33]:
from FPL_Optimizer import FPLOptimizer

In [34]:
print(f"\nPredicted Ownership Weighted xPoints between {current_gw+1} and {current_gw+5}:")
if __name__ == "__main__":
    custom_columns = {
        'player_name': 'player_name_id',
        'team_name': 'team_name',
        'position': 'position',
        'value': 'value',
        'xpoints': 'blended_score'
    }
    
    optimizer = FPLOptimizer(
        df=ownership_weighted_predictions,
        budget=101.0,
        column_mapping=custom_columns
    )
    optimizer.optimize()
    
    # Get results as DataFrames
    squad = optimizer.get_squad_dataframe()
    starting_xi = optimizer.get_starting_xi_dataframe()


Predicted Ownership Weighted xPoints between 10 and 14:
FPL TEAM OPTIMIZER (Linear Programming)

Budget for 15-Player Squad: £101.0m

Solving optimization problem...

OPTIMAL 15-PLAYER SQUAD
Starting XI Formation: 3-5-2
Total Squad Cost: £96.2m / £101.0m
Total Weighted Expected Points: 1605.25
Remaining Budget: £4.8m

--------------------------------GOALKEEPERS (2)---------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Melker Ellborg                 Sunderland   £4.4   112.63 
 ✅        Freddie Woodman                Liverpool    £4.0   113.31 

---------------------------------DEFENDERS (5)----------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Benjamin White                 Arsenal      £5.4   132.27 
 🪑       

In [35]:
print(f"\nPredicted xPoints between {current_gw+1} and {current_gw+5}:")
next5gw_predictions = db_creator.run_sql(f"""
    WITH predictions_status AS (
    SELECT *
    FROM predictions
    LEFT JOIN player_status ON predictions.element = player_status.id
    )
    SELECT
        player_name_id,
        status,
        position,
        team_name,
        now_cost / 10.0 AS value,
        ROUND(AVG(opp_team_elo),1) AS avg_opp_team_elo,
        ROUND(SUM(xpoints),2) AS total_xpoints,
        ROUND(AVG(mins_pred),1) AS avg_mins_pred,
        ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
        ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
        ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
    FROM predictions_status
    WHERE event BETWEEN {current_gw+1} AND {current_gw+5}
        AND status IN ('a', 'd')
        -- AND position = 'FWD'
        -- AND value <= 45
        -- AND team_name = 'Crystal Palace'
    GROUP BY player_name_id, status, position, team_name, now_cost
    ORDER BY total_xpoints DESC
""").dropna(subset=['value', 'total_xpoints'])

# Usage
if __name__ == "__main__":
    custom_columns = {
        'player_name': 'player_name_id',
        'team_name': 'team_name',
        'position': 'position',
        'value': 'value',
        'xpoints': 'total_xpoints'
    }
    
    optimizer = FPLOptimizer(
        df=next5gw_predictions,
        budget=101.0,
        column_mapping=custom_columns
    )
    optimizer.optimize()
    
    # Get results as DataFrames
    squad = optimizer.get_squad_dataframe()
    starting_xi = optimizer.get_starting_xi_dataframe()


Predicted xPoints between 10 and 14:
FPL TEAM OPTIMIZER (Linear Programming)

Budget for 15-Player Squad: £101.0m

Solving optimization problem...

OPTIMAL 15-PLAYER SQUAD
Starting XI Formation: 3-4-3
Total Squad Cost: £100.7m / £101.0m
Total Weighted Expected Points: 289.97
Remaining Budget: £0.3m

--------------------------------GOALKEEPERS (2)---------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Freddie Woodman                Liverpool    £4.0    19.68 
 ✅        Marco Bizot                    Aston Villa  £4.5    19.81 

---------------------------------DEFENDERS (5)----------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Ezri Konsa Ngoyo               Arsenal      £4.7    22.51 
 🪑        Issa Diop         

In [36]:
remaining_gw_predictions = db_creator.run_sql(f"""
    WITH predictions_status AS (
    SELECT *
    FROM predictions
    LEFT JOIN player_status ON predictions.element = player_status.id
    )
    SELECT
        event,
        player_name_id,
        status,
        position,
        team_name,
        opp_team_name,
        now_cost / 10.0 AS value,
        xpoints
    FROM predictions_status
    WHERE event BETWEEN {current_gw+1} AND 38
        AND status IN ('a', 'd')
    ORDER BY xpoints DESC
""")
print("Best week to triple captain:")
remaining_gw_predictions.head(10)

Best week to triple captain:


,event,player_name_id,status,position,team_name,opp_team_name,value,xpoints
0,21.0,Florian Wirtz,a,MID,Liverpool,Crystal Palace,7.2,6.052136
1,10.0,Florian Wirtz,a,MID,Liverpool,Crystal Palace,7.2,6.052136
2,19.0,Florian Wirtz,a,MID,Liverpool,Coventry City,7.2,6.033948
3,25.0,Florian Wirtz,a,MID,Liverpool,Coventry City,7.2,6.033948
4,20.0,Florian Wirtz,a,MID,Liverpool,Sunderland,7.2,6.017328
5,13.0,Florian Wirtz,a,MID,Liverpool,Sunderland,7.2,6.017328
6,29.0,Florian Wirtz,a,MID,Liverpool,Ipswich Town,7.2,6.012809
7,17.0,Florian Wirtz,a,MID,Liverpool,Hull City,7.2,5.973121
8,26.0,Florian Wirtz,a,MID,Liverpool,Hull City,7.2,5.973121
9,16.0,Florian Wirtz,a,MID,Liverpool,Spurs,7.2,5.969939


In [37]:
print(f"\nOptimizing team selection for GW{current_gw+1} to GW38...")

if __name__ == "__main__":
    custom_columns = {
        'player_name': 'player_name_id',
        'team_name': 'team_name',
        'position': 'position',
        'value': 'value',
        'xpoints': 'xpoints'
    }
    all_teams =pd.DataFrame()
    for event in range(current_gw+1, 19): #change to 39 for 2nd half of season
        print(f"\nOptimizing team selection for GW{event}...")
        try:
            optimizer = FPLOptimizer(
                df=remaining_gw_predictions[remaining_gw_predictions['event']==event],
                budget=100.0,
                column_mapping=custom_columns
            )
            optimizer.optimize()
        
            # Get results as DataFrames
            # squad = optimizer.get_squad_dataframe()
            starting_xi = optimizer.get_starting_xi_dataframe()
            # result = optimizer.calculate_bench_boost_xpoints()
            starting_xi['event'] = event
            all_teams = pd.concat([all_teams, starting_xi], ignore_index=True)
        except Exception as e:
            print(f"Error optimizing for GW{event}: {e}")


Optimizing team selection for GW10 to GW38...

Optimizing team selection for GW10...
FPL TEAM OPTIMIZER (Linear Programming)

Budget for 15-Player Squad: £100.0m

Solving optimization problem...

OPTIMAL 15-PLAYER SQUAD
Starting XI Formation: 3-4-3
Total Squad Cost: £99.9m / £100.0m
Total Weighted Expected Points: 58.96
Remaining Budget: £0.1m

--------------------------------GOALKEEPERS (2)---------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Freddie Woodman                Liverpool    £4.0     4.12 
 ✅        David Raya Martín              Arsenal      £6.2     4.37 

---------------------------------DEFENDERS (5)----------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Rico Lewis                     Man City  

In [38]:
freehit_points = all_teams.pivot_table(index='event', values='xpoints', aggfunc='sum').sort_values(by='xpoints', ascending=False)
best_week = freehit_points.head(1)['xpoints'].index.astype(int).values[0]
print(f"Best week to freehit: {best_week}, points: {round(all_teams[all_teams['event']==best_week]['xpoints'].sum(),1)}")
all_teams[all_teams['event']==best_week].sort_values(by=['position','xpoints'], ascending=False)

Best week to freehit: 17, points: 57.6


,event,player_name_id,status,position,team_name,opp_team_name,value,xpoints
77,17,Florian Wirtz,a,MID,Liverpool,Hull City,7.2,5.973121
79,17,Morgan Gibbs-White,a,MID,Nott'm Forest,Man Utd,7.9,5.359261
80,17,Declan Rice,d,MID,Arsenal,Crystal Palace,7.4,5.346943
81,17,Martin Ødegaard,a,MID,Arsenal,Crystal Palace,6.7,5.307445
87,17,Freddie Woodman,a,GK,Liverpool,Hull City,4.0,4.179804
78,17,Erling Haaland,a,FWD,Man City,Newcastle,15.8,5.624201
82,17,Nicolas Jackson,a,FWD,Aston Villa,Leeds,7.0,5.275725
84,17,Francisco Evanilson de Lima Barbosa,a,FWD,Bournemouth,Spurs,6.4,5.201200
83,17,Cristhian Mosquera,a,DEF,Arsenal,Crystal Palace,5.2,5.236641
85,17,António João Pereira de Albuquerque Tavares da...,a,DEF,Bournemouth,Spurs,5.0,5.071218


In [39]:
# finish=time.perf_counter()
# print(f'Finished in {round(((finish-start)/60),2)} minute(s)') 
# #Finished in 0.01 minute(s)

In [40]:
# Notification when file has finished running
from plyer import notification

notification.notify(
    title='Script Complete',
    message='Your file has finished running!',
    app_name='Jupyter Notebook',
    timeout=20  # notification stays for 20 seconds
)